In [1]:
import dijet
from itertools import product
import plot_dsa_tools as tools
import importlib
importlib.reload(tools)

import numpy as np
import pandas as pd

In [13]:
# get data

fold = 'data/largeNcq'
zfiles = [f'{fold}/dsa_pt0p1z0p2_rs40_Q00p5_xi1p5_pn_200reps_largeNcq.npy',
         f'{fold}/dsa_pt0p1z0p5_rs40_Q00p5_xi1p5_pn_200reps_largeNcq.npy']
# zfiles = [f'{fold}/dsa_pt0p1z0p2_rs40_Q00p5_xi1p5_pn_300reps_largeNcq_constrained.npy',
#          f'{fold}/dsa_pt0p1z0p5_rs40_Q00p5_xi1p5_pn_300reps_largeNcq_constrained.npy']
zdata = tools.get_data(zfiles)

fold = 'data/largeNcq'
tfile = [f'{fold}/dsa_pt0p1_rs40_Q00p5_xi1p5_pn_200reps_largeNcq.npy']
tdata = tools.get_data(tfile)


def get_number(i, target, obs, lumi):
    num = 0 
    if i == 0: num += 1000
    # elif i == 1: num += 2000
    elif i == 1: num += 11000
    else: raise ValueError(f'do not know i={i}')

    if target == 'p': num += 0
    elif target == 'd': num += 100
    elif target == 'h': num += 200
    else: raise ValueError(f'do not know tar={target}')

    if obs == 'ALL': num += 0
    elif obs == 'cos(phi_kp)': num += 10
    elif obs == 'cos(phi_Dp)': num += 20
    elif obs == 'cos(phi_Dp)cos(phi_kp)': num += 30
    elif obs == 'sin(phi_Dp)sin(phi_kp)': num += 40
    else: raise ValueError(f'do not know ob={obs}')

    if lumi == 100: num += 0
    elif lumi == 10: num += 1
    else: raise ValueError(f'do not know lumi={lumi}')

    return num

In [14]:
ys = [0.05, 0.95]
Q2s = [1, 100]
zs = [0.2, 0.5]
ts = 0.1
pTs = zdata['data'][0]['pT values']
roots = 40
lumis = [10, 100]
nreps = len(zdata['data'][0]['p'])

pTs_data = np.arange(2, 10.5, 0.5)
print('pT values:', pTs_data)

obs = ['ALL', 'cos(phi_kp)', 'cos(phi_Dp)', 'cos(phi_Dp)cos(phi_kp)', 'sin(phi_Dp)sin(phi_kp)']
targets = ['p', 'd', 'h']

for lumi in lumis:
    for idat, data in enumerate([tdata, zdata]):
        if idat == 0: continue # testing: don't produce tdata
        
        if idat == 0: 
            header = ['Ymin', 'Ymax', 'Q2min', 'Q2max', 'Zmin', 'Zmax', 'PT', 'T', 'RS', 'obs', 'lum', 'value', 'stat_u', 'sys_u', 'tar', 'norm_c']
        else: 
            header = ['Ymin', 'Ymax', 'Q2min', 'Q2max', 'PT', 'T', 'Z', 'RS', 'obs', 'lum', 'value', 'stat_u', 'sys_u', 'tar', 'norm_c']

        for tar in targets:
            for ob in obs:
                errmod = 1
                if ob == 'ALL': 
                    obl = '1'
                else: 
                    obl = ob
                    errmod = 2

                if tar == 'd':
                    p,n = 1,1
                    omega_d = 0.07
                    Pp = 1-1.5*omega_d
                    Pn = Pp
                    errmod *= (Pp**2 + Pn**2)/((p+n)**2)
            
                elif tar == 'h':
                    p,n = 2,1
                    pS,pD,pSp = 0.9,0.1,0.02
                    Pp = -(4./3.)*(pD-pSp)
                    Pn = pS - (1./3.)*(pD-pSp)
                    errmod *= (Pp**2 + Pn**2)/((p+n)**2)

                pseudodata = []
                if idat == 1:
                    for (iz,z), (ipT, pT) in product(enumerate(zs), enumerate(pTs)):
                        if pT not in pTs_data: continue
                        if pT < 5: continue
                        value = np.mean([data['data'][iz][tar][rep][f'<{obl}>'][ipT] for rep in range(nreps)])
                        # value = data['data'][iz][tar][1][f'<{obl}>'][ipT]

                        if z == 0.5 and ob in ['cos(phi_kp)', 'cos(phi_Dp)']: continue

                        stat_error = np.sqrt(errmod/(data['data'][iz]['p'][0]['denom'][ipT]*lumi))
                        stat_error *= 1/0.7 # avg. pol. of proton for EIC (see 1212.1701)
                        stat_error *= 1/0.8 # avg. pol. of electron for EIC (see 1212.1701)
                        sys_error = 0.05*value
                        pseudodata.append([ys[0], ys[1], Q2s[0], Q2s[1], pT, ts, z, roots, ob, lumi, value, stat_error, sys_error, tar, 1.0])
                else:
                    for ipT, pT in enumerate(pTs):
                        if pT not in pTs_data: continue
                        
                        value = np.mean([data['data'][0][tar][rep][f'<{obl}>'][ipT] for rep in range(nreps)])
                        value = data['data'][0][tar][1][f'<{obl}>'][ipT]
                        stat_error = np.sqrt(errmod/(data['data'][0]['p'][0]['denom'][ipT]*lumi))
                        stat_error *= 1/0.7 # avg. pol. of proton for EIC (see 1212.1701)
                        stat_error *= 1/0.8 # avg. pol. of electron for EIC (see 1212.1701)
                        sys_error = 0.05*value
                        pseudodata.append([ys[0], ys[1], Q2s[0], Q2s[1], zs[0], zs[1], pT, ts, roots, ob, lumi, value, stat_error, sys_error, tar, 1.0])

                sheet_num = get_number(idat, tar, ob, lumi)
                filename = f'{sheet_num}.xlsx'
                print(idat, tar, ob, lumi, filename)
                df = pd.DataFrame(pseudodata, columns=header)
                df.to_excel('pseudodata/'+filename, index=False)
        


pT values: [ 2.   2.5  3.   3.5  4.   4.5  5.   5.5  6.   6.5  7.   7.5  8.   8.5
  9.   9.5 10. ]
1 p ALL 10 11001.xlsx
1 p cos(phi_kp) 10 11011.xlsx
1 p cos(phi_Dp) 10 11021.xlsx
1 p cos(phi_Dp)cos(phi_kp) 10 11031.xlsx
1 p sin(phi_Dp)sin(phi_kp) 10 11041.xlsx
1 d ALL 10 11101.xlsx
1 d cos(phi_kp) 10 11111.xlsx
1 d cos(phi_Dp) 10 11121.xlsx
1 d cos(phi_Dp)cos(phi_kp) 10 11131.xlsx
1 d sin(phi_Dp)sin(phi_kp) 10 11141.xlsx
1 h ALL 10 11201.xlsx
1 h cos(phi_kp) 10 11211.xlsx
1 h cos(phi_Dp) 10 11221.xlsx
1 h cos(phi_Dp)cos(phi_kp) 10 11231.xlsx
1 h sin(phi_Dp)sin(phi_kp) 10 11241.xlsx
1 p ALL 100 11000.xlsx
1 p cos(phi_kp) 100 11010.xlsx
1 p cos(phi_Dp) 100 11020.xlsx
1 p cos(phi_Dp)cos(phi_kp) 100 11030.xlsx
1 p sin(phi_Dp)sin(phi_kp) 100 11040.xlsx
1 d ALL 100 11100.xlsx
1 d cos(phi_kp) 100 11110.xlsx
1 d cos(phi_Dp) 100 11120.xlsx
1 d cos(phi_Dp)cos(phi_kp) 100 11130.xlsx
1 d sin(phi_Dp)sin(phi_kp) 100 11140.xlsx
1 h ALL 100 11200.xlsx
1 h cos(phi_kp) 100 11210.xlsx
1 h cos(phi_Dp) 1

In [ ]:
ys = [0.05, 0.95]
Q2s = [1, 100]
zs = [0.2, 0.5]
ts = [0.1]
pTs = data['data'][0]['pT values']
rss = [40]
lumi = 100
nreps = len(data['data'][0]['p'])

obs = ['ALL', 'cos(phi_kp)', 'cos(phi_Dp)', 'cos(phi_Dp)cos(phi_kp)', 'sin(phi_Dp)sin(phi_kp)']
header = ['Ymin', 'Ymax', 'Q2min', 'Q2max', 'Zmin', 'Zmax', 'PT', 'T', 'RS', 'obs', 'lum', 'value', 'stat_u', 'sys_u', 'tar', 'norm_c']
targets = ['p', 'd', 'h']

for tar in targets:
    print(tar)
    pseudodata = []
    for ob in obs:
        errmod = 1
        if ob == 'ALL': 
            obl = '1'
        else: 
            obl = ob
            errmod = 2

        if tar == 'd':
            p,n = 1,1
            omega_d = 0.07
            Pp = 1-1.5*omega_d
            Pn = Pp
            errmod *= (Pp**2 + Pn**2)/((p+n)**2)

        elif tar == 'h':
            p,n = 2,1
            pS,pD,pSp = 0.9,0.1,0.02
            Pp = -(4./3.)*(pD-pSp)
            Pn = pS - (1./3.)*(pD-pSp)
            errmod *= (Pp**2 + Pn**2)/((p+n)**2)
        
        for rs, t, (ipT, pT) in product(rss, ts, enumerate(pTs)):
            # if pT > 5: continue
            # if pT < 2: continue
            if pT not in pTs_data: continue
            
            value = np.mean([data['data'][0][tar][rep][f'<{obl}>'][ipT] for rep in range(nreps)])
            stat_error = np.sqrt(errmod/(data['data'][0]['p'][0]['denom'][ipT]*lumi))
            stat_error *= 1/0.7 # avg. pol. of proton for EIC (see 1212.1701)
            stat_error *= 1/0.8 # avg. pol. of electron for EIC (see 1212.1701)
            sys_error = 0.05*value
            pseudodata.append([ys[0], ys[1], Q2s[0], Q2s[1], zs[0], zs[1], pT, t, rs, ob, lumi, value, stat_error, sys_error, tar, 1.0])

    df = pd.DataFrame(pseudodata, columns=header)
    if lumi == 100:
        filenames = {'p': f'4100.xlsx', 'd': f'4101.xlsx', 'h': f'4102.xlsx'}
    elif lumi == 10:
        filenames = {'p': f'4010.xlsx', 'd': f'4011.xlsx', 'h': f'4012.xlsx'}
    else:
        raise ValueError(f'lumi {lumi} is not 10 or 100')
    print(filenames[tar])
    df.to_excel('pseudodata/'+filenames[tar], index=False)
        

p
4100.xlsx
d
4101.xlsx
h
4102.xlsx
